# Assessing the uncertainties in the primary-gamma rays (our signal)

The MC samples have follower particle tracking enabled, which records gamma-rays emitted during the primary (initial) interaction and any secondaries downstream (from Geant4 neutron/proton-nucleus interactions)

The uncertainties are taken from our Empirical model for the p and s-shell states, and applied to signal and (some) background events. Because no GENIE re-weighting exists for the de-excitation model, this is a little ad-hoc but the best I could do.

Why we must apply it to both signal and background:

You might think to yourself "signal should be the only affected population as the nuclear de-excitations we're tagging are coming from NCQE events" - while this is the case a large % of one background category in our sample (out-of-fiducial-volume / OOFV) events contain NCQE events that were mis-reconstructed as occuring within our FV. We therefore must apply the variation to both signal + OOFV background events.

In [3]:
from lib.constants import *
from lib.imports import *

# Any aggregate MC sample will do (with the follower gamma-ray information)
# It's much easier to pair clusters : events with aggregate files that have been processed through xsec_analyzer
file_name = '../GENIE-Empirical-4.417e20-ntuple_dirt_downsampled.root'

# scalar accumulators
MC_Vtx               = []
MC_Vty               = []
MC_Vtz               = []
MC_Vtt               = []
MC_External          = []
MC_FV                = []
MC_NC                = []
MC_QEL               = []
MC_MEC               = []
MC_PDG               = []
MC_Z                 = []
MC_Z_nucleon         = []
MC_remnant_nucleus  = []

# jagged accumulators
MC_FollowerT         = []
MC_FollowerE         = []
MC_FollowerPDG       = []
MC_FollowerParentPDG = []

MC_SIGNAL = []          # 0 or 1 for NCQE true signal
MC_EVENT_CATEGORY = []  # see categories below
'''
// 0 = nuNCQE
// 1 = External
// 2 = Out-of-FV
// 3 = CC
// 4 = NCother (includes MEC, NC RES, NC DIS, NC on Hydrogen (which may be labeled as QEL), NC EL)
// 5 = nubarNCQE
'''
MC_SELECTED = []        # 0 or 1 for reconstructed
    
root = uproot.open(file_name)

# ── trigger tree ──────────────────────────────────────────────────────────
S   = root['phaseIITriggerTree']
vx  = S['trueNuIntxVtx_X'].array(library='np')
vy  = S['trueNuIntxVtx_Y'].array(library='np')
vz  = S['trueNuIntxVtx_Z'].array(library='np')

vx_cm = vx / 100
vy_cm = vy / 100 + 0.1446
vz_cm = vz / 100 - 1.681
r2    = vx_cm**2 + vz_cm**2
ext_flags = (r2 > radius**2)    | (np.abs(vy_cm) > half_height)
fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height)

# map trigger-level quantities onto clusters via CEN
MC_Vtx.append(vx_cm)
MC_Vty.append(vy_cm)
MC_Vtz.append(vz_cm)
MC_Vtt.append(S['trueNuIntxVtx_T'].array(library='np'))
MC_External.append(ext_flags)
MC_FV.append(fv_flags)
MC_NC.append(S['trueNC'].array(library='np'))
MC_QEL.append(S['trueQEL'].array(library='np'))
MC_MEC.append(S['trueMEC'].array(library='np'))
MC_PDG.append(S['trueFSLPdg'].array(library='np'))
MC_Z.append(S['trueTargetZ'].array(library='np'))
MC_Z_nucleon.append(S['trueTargetNucleonPDG'].array(library='np'))    # proton = 2212, neutron = 2112
MC_remnant_nucleus.append(S['trueRemnantNucleusPDG'].array(library='np'))

MC_FollowerT.append(S['trueFollowerStartT'].array(library='ak'))
MC_FollowerE.append(S['trueFollowerE'].array(library='ak'))
MC_FollowerPDG.append(S['trueFollowerPDG'].array(library='ak'))
MC_FollowerParentPDG.append(S['trueFollowerParentPDG'].array(library='ak'))

MC_SIGNAL.append(S['NCQE_MC_Signal'].array(library='np'))
MC_EVENT_CATEGORY.append(S['NCQE_EventCategory'].array(library='np'))
MC_SELECTED.append(S['NCQE_Selected'].array(library='np'))
    
    
# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

MC_Vtx             = np.concatenate(MC_Vtx)
MC_Vty             = np.concatenate(MC_Vty)
MC_Vtz             = np.concatenate(MC_Vtz)
MC_Vtt             = np.concatenate(MC_Vtt)
MC_External        = np.concatenate(MC_External)
MC_FV              = np.concatenate(MC_FV)
MC_NC              = np.concatenate(MC_NC)
MC_QEL             = np.concatenate(MC_QEL)
MC_MEC             = np.concatenate(MC_MEC)
MC_PDG             = np.concatenate(MC_PDG)
MC_Z               = np.concatenate(MC_Z)
MC_Z_nucleon       = np.concatenate(MC_Z_nucleon)
MC_remnant_nucleus = np.concatenate(MC_remnant_nucleus)

MC_FollowerT         = ak.concatenate(MC_FollowerT)
MC_FollowerE         = ak.concatenate(MC_FollowerE)
MC_FollowerPDG       = ak.concatenate(MC_FollowerPDG)
MC_FollowerParentPDG = ak.concatenate(MC_FollowerParentPDG)

MC_SIGNAL            = np.concatenate(MC_SIGNAL)
MC_EVENT_CATEGORY    = np.concatenate(MC_EVENT_CATEGORY)
MC_SELECTED          = np.concatenate(MC_SELECTED)


print(f'Total events: {len(MC_Vtt)}')
print('\ndone')


Managing arrays...

Total events: 2358240

done


In [6]:
"""
═══════════════════════════════════════════
  Branching-ratio systematic propagation
═══════════════════════════════════════════
Ensure the following are in scope:
    MC_Vtx, MC_Vty, MC_Vtz, MC_Vtt, MC_External, MC_FV, MC_NC, MC_QEL,
    MC_MEC, MC_PDG, MC_Z, MC_Z_nucleon, MC_remnant_nucleus   (numpy)
    MC_FollowerT, MC_FollowerE, MC_FollowerPDG, MC_FollowerParentPDG (awkward)
    MC_SIGNAL, MC_EVENT_CATEGORY, MC_SELECTED                (numpy)
"""

# ═══════════════════════════════════════════════════════════════════════════
#  CELL 1 -- Masks
# ═══════════════════════════════════════════════════════════════════════════

# Truth-level FV NCQE population - OUR SIGNAL (used only as the R_eff denominator --
# the total true signal population, selected or not).
mask_fv_truth = (
    (MC_FV       == True) &
    (MC_External == False) &
    (MC_NC  == 1) &
    (MC_QEL == 1) &
    (MC_Z   == 8) &
    ((MC_PDG == 14) | (MC_PDG == 12))
)

# Reconstructed, selected FV NCQE signal -- this IS N' directly; no separate
# truth-level classification pass is needed anymore.
mask_signal_reco = (
    mask_fv_truth &
    (MC_SIGNAL         == 1) &
    (MC_EVENT_CATEGORY == 0) &
    (MC_SELECTED       == 1)
)

# Truth-level OOFV NCQE population (parallel structure to mask_fv_truth).
mask_oofv_truth = (
    (MC_FV       == False) &
    (MC_External == False) &
    (MC_NC  == 1) &
    (MC_QEL == 1) &
    (MC_Z   == 8) &
    ((MC_PDG == 14) | (MC_PDG == 12))
)

# Reconstructed, selected OOFV background -- the only background category
# in scope per your instruction.
mask_bkg_reco = (
    mask_oofv_truth &
    (MC_EVENT_CATEGORY == 2) &
    (MC_SELECTED       == 1)
)

N_total_truth_FV = int(np.sum(mask_fv_truth))
N_sig_TOTAL_reco  = int(np.sum(mask_signal_reco))
N_bkg_TOTAL_reco  = int(np.sum(mask_bkg_reco))

print(f"N_total (truth FV NCQE)              = {N_total_truth_FV}")
print(f"N' (reco-selected FV signal)         = {N_sig_TOTAL_reco}")
print(f"N_bkg (reco-selected OOFV background) = {N_bkg_TOTAL_reco}")


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 2 -- Thresholds and ideal-model line lists
# ═══════════════════════════════════════════════════════════════════════════
# These conditions are meant to select "primary" gamma-rays, and have been
# optimized looking at follower particle distributions

T_PRIMARY_NS = 0.001    # |dt| < 1 ps -> primary de-excitation gamma
E_LOW_MEV    = 3.0
E_MID_MEV    = 6.0
E_TOL_MEV    = 0.0005   # tightened match tolerance; loosen if diagnostic
                        # below shows larger float drift in this ntuple

FOLLOWER_E_ALREADY_MEV = True   # enable if followers are in units of MeV (they should be)

# Extracted from levels O16 de-excitation (see Steven Doran's thesis)
P_SHELL_LINES_MEV = [6.32, 9.93, 3.61, 6.18]
S_SHELL_LINES_MEV = [
    3.09, 3.38, 3.68, 3.85, 3.89, 4.44, 4.92, 5.11, 5.69, 5.83,
    6.09, 6.20, 6.44, 6.45, 6.59, 6.73, 6.90, 7.01, 7.03, 7.34,
]
_overlap = [(p, s) for p in P_SHELL_LINES_MEV for s in S_SHELL_LINES_MEV
            if abs(p - s) < E_TOL_MEV]
assert not _overlap, f"p-shell/s-shell line collision: {_overlap}"


# Break up the gammas into different bands as Kobayashi 2006 applies systematics
# in a 3 > E_gamma > 6 MeV, E_gamma > 6 MeV manner
def classify_primary_gamma(E_MeV):
    for line in P_SHELL_LINES_MEV:
        if abs(E_MeV - line) < E_TOL_MEV:
            return ('p', 'hi' if E_MeV > E_MID_MEV else 'mid')
    for line in S_SHELL_LINES_MEV:
        if abs(E_MeV - line) < E_TOL_MEV:
            return ('s', 'hi' if E_MeV > E_MID_MEV else 'mid')
    return None


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 3 -- Event classification, WITH immune-secondary awareness
# ═══════════════════════════════════════════════════════════════════════════
# See Steven Doran's thesis for event categorization following an NCQE interaction
#
# Prompt NCQE topologies can be split into 4 categories:
# 1. Primary gamma only
# 2. Primary and secondary gamma rays
# 3. Secondary gamma ray only
# 4. Neither
#
# Use cat1/cat2/cat3/cat4 distinction from the original R_eff as outlined in Steven's
# thesis analysis. A primary-gamma match alone is NOT sufficient to call an
# event "at risk" under the branching-ratio nuisance parameters -- an event
# that also carries an immune secondary gamma (from FSI of the knocked-out
# nucleon on a neighboring oxygen nucleus) would stay selected even if the
# primary gamma had not fired. Only cat1 events (primary, no immune
# secondary) are genuinely sensitive to this systematic.
#
#   cat1 = has_primary & ~is_immune   -> AT RISK (branching-ratio shiftable)
#   cat2 = has_primary &  is_immune   -> immune (primary + secondary)
#   cat3 = ~has_primary & is_immune   -> immune (secondary only)
#   cat4 = ~has_primary & ~is_immune  -> no gamma signal of any kind
#                                        (We see a lot of these, which I think is due to some tracking issue with the particles)
#                                        (Since we don't have access to those events we ignore them --> not the best but its what I can do with the files I have available)

T_NCAP_NS    = 100.0   # secondary-gamma upper timing bound (excludes
                        # neutron capture)
E_IMMUNE_MEV = 3.0     # Option A: summed secondary energy > this MeV makes
                        # an event immune. 3 MeV is a reasonable detection floor
                         # any secondary gamma-rays below this likely would not be immune,
                          # and rely exclusively on the primary gamma ray to tag


def is_immune_event(E_MeV_all, par, pdg, t, T_prim=T_PRIMARY_NS,
                     T_ncap=T_NCAP_NS, E_immune=E_IMMUNE_MEV):
    sec = (((par == 2112) | (par == 2212)) & (pdg == 22)
           & (t > T_prim) & (t < T_ncap))
    E_sec = E_MeV_all[sec]
    if E_immune == 0.0:
        return bool(np.any(sec))
    return bool(np.sum(E_sec) > E_immune)


def classify_events_with_immunity(f_energy, f_parent_pdg, f_pdg, f_time,
                                    T_prim=T_PRIMARY_NS):
    N = len(f_energy)
    has_primary   = np.zeros(N, bool)
    is_immune     = np.zeros(N, bool)
    event_classes = [[] for _ in range(N)]
    n_unmatched   = 0

    for i in range(N):
        E   = np.asarray(f_energy[i],     dtype=float)
        par = np.asarray(f_parent_pdg[i], dtype=int)
        pdg = np.asarray(f_pdg[i],        dtype=int)
        t   = np.asarray(f_time[i],       dtype=float)

        E_MeV_all = E if FOLLOWER_E_ALREADY_MEV else E * 1000.0

        prim = (par == 0) & (pdg == 22) & (np.abs(t) < T_prim)
        E_prim_MeV = E_MeV_all[prim]

        if E_prim_MeV.size > 0:
            has_primary[i] = True
            for E_g in E_prim_MeV:
                cls = classify_primary_gamma(E_g)
                if cls is None:
                    n_unmatched += 1
                    continue
                event_classes[i].append(cls)

        is_immune[i] = is_immune_event(E_MeV_all, par, pdg, t, T_prim)

    if n_unmatched:
        print(f"  WARNING: {n_unmatched} primary photon(s) did not match "
              f"any known ideal-model line (energy-matching issue, "
              f"distinct from immunity).")

    return has_primary, is_immune, event_classes


def category_counts(has_primary, is_immune):
    return dict(
        cat1=int(np.sum(has_primary & ~is_immune)),
        cat2=int(np.sum(has_primary &  is_immune)),
        cat3=int(np.sum(~has_primary & is_immune)),
        cat4=int(np.sum(~has_primary & ~is_immune)),
    )


def dominant_class_cat1_only(has_primary, is_immune, event_classes):
    """Dominant (shell, band) class -- ONLY for cat1 (at-risk) events.
    Everything else (cat2, cat3, cat4) gets None, i.e. held fixed under any
    branching-ratio nuisance shift."""
    N = len(has_primary)
    out = []
    for i in range(N):
        at_risk = has_primary[i] and not is_immune[i]
        if not at_risk or not event_classes[i]:
            out.append(None)
            continue
        bands = event_classes[i]
        if any(b == 'hi' for _, b in bands):
            shell = next(s for s, b in bands if b == 'hi')
            out.append((shell, 'hi'))
        else:
            out.append(bands[0])
    return np.array(out, dtype=object)


CLASS_LABELS = [('p', 'hi'), ('p', 'mid'), ('s', 'hi'), ('s', 'mid')]

def class_counts(dominant_class):
    return {
        cls: int(np.sum([d == cls for d in dominant_class if d is not None]))
        for cls in CLASS_LABELS
    }


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 4 -- Classify N' (reconstructed signal) and N_bkg (reco background),
#            splitting each into cat1 (at-risk) vs cat2+cat3+cat4 (immune)
# ═══════════════════════════════════════════════════════════════════════════
def build_follower_slices(mask):
    return (
        MC_FollowerE[mask],
        MC_FollowerParentPDG[mask],
        MC_FollowerPDG[mask],
        MC_FollowerT[mask] - MC_Vtt[mask],
    )

# --- signal (N') -------------------------------------------------------------
e_s, pp_s, p_s, t_s = build_follower_slices(mask_signal_reco)
has_primary_sig, is_immune_sig, event_classes_sig = classify_events_with_immunity(
    e_s, pp_s, p_s, t_s
)
cats_sig = category_counts(has_primary_sig, is_immune_sig)
dominant_sig = dominant_class_cat1_only(has_primary_sig, is_immune_sig, event_classes_sig)
N_class_signal = class_counts(dominant_sig)          # cat1 only, by class
N_sig_at_risk   = sum(N_class_signal.values())        # = cat1 total
N_sig_immune    = N_sig_TOTAL_reco - N_sig_at_risk    # cat2 + cat3 + cat4,
                                                        # held fixed

# --- background (N_bkg) -------------------------------------------------------
e_b, pp_b, p_b, t_b = build_follower_slices(mask_bkg_reco)
has_primary_bkg, is_immune_bkg, event_classes_bkg = classify_events_with_immunity(
    e_b, pp_b, p_b, t_b
)
cats_bkg = category_counts(has_primary_bkg, is_immune_bkg)
dominant_bkg = dominant_class_cat1_only(has_primary_bkg, is_immune_bkg, event_classes_bkg)
N_class_bkg = class_counts(dominant_bkg)
N_bkg_at_risk = sum(N_class_bkg.values())
N_bkg_immune  = N_bkg_TOTAL_reco - N_bkg_at_risk

N_sig_unmatched = N_sig_immune
N_bkg_unmatched = N_bkg_immune

print("\n" + "=" * 70)
print("  CATEGORY BREAKDOWN (cat1 = at risk; cat2+cat3+cat4 = immune)")
print("=" * 70)
print(f"  {'':<8}{'cat1':>8}{'cat2':>8}{'cat3':>8}{'cat4':>8}{'dilution':>12}")
for label, cats, n_tot in [('signal', cats_sig, N_sig_TOTAL_reco),
                            ('bkg',    cats_bkg, N_bkg_TOTAL_reco)]:
    dilution = 100 * (cats['cat2'] + cats['cat3']) / n_tot if n_tot else 0.0
    print(f"  {label:<8}{cats['cat1']:>8}{cats['cat2']:>8}"
          f"{cats['cat3']:>8}{cats['cat4']:>8}{dilution:>11.1f}%")

print("\n" + "=" * 70)
print("  CLASSIFIED COUNTS (cat1 / at-risk population only)")
print("=" * 70)
print(f"  {'Class':<12}{'N (signal)':>12}{'N (bkg)':>12}")
for cls in CLASS_LABELS:
    print(f"  {cls[0]}-shell {cls[1]:<4}"
          f"{N_class_signal[cls]:>12d}{N_class_bkg[cls]:>12d}")
print(f"  {'immune':<12}{N_sig_immune:>12d}{N_bkg_immune:>12d}")
print(f"\n  NOTE: immune events (cat2+cat3+cat4) are held FIXED -- no "
      f"branching-ratio sigma applied -- since their selection does not "
      f"depend on the primary de-excitation gamma. Only cat1 counts above "
      f"move under the nuisance shifts below.")


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 5 -- Relative uncertainties per class (Leuschner 1994 CCIA / p-shell,
#            Kobayashi 2006 / s-shell), stat+syst in quadrature
# ═══════════════════════════════════════════════════════════════════════════
sigma_p_up   = np.sqrt(0.8**2 + 5.4**2)    / 100.0
sigma_p_down = np.sqrt(0.8**2 + 16.4**2)   / 100.0

S_MID_CENTRAL = 27.9
sigma_s_mid_up   = np.sqrt(1.5**2 + 3.4**2) / S_MID_CENTRAL
sigma_s_mid_down = np.sqrt(1.5**2 + 2.6**2) / S_MID_CENTRAL

S_HI_CENTRAL = 15.6
sigma_s_hi_up   = np.sqrt(1.3**2 + 0.6**2) / S_HI_CENTRAL
sigma_s_hi_down = np.sqrt(1.3**2 + 1.0**2) / S_HI_CENTRAL

SIGMA_REL_UP = {
    ('p', 'hi'):  sigma_p_up,  ('p', 'mid'): sigma_p_up,
    ('s', 'hi'):  sigma_s_hi_up, ('s', 'mid'): sigma_s_mid_up,
}
SIGMA_REL_DOWN = {
    ('p', 'hi'):  sigma_p_down,  ('p', 'mid'): sigma_p_down,
    ('s', 'hi'):  sigma_s_hi_down, ('s', 'mid'): sigma_s_mid_down,
}


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 6 -- Correlated propagation onto the extracted signal count
# ═══════════════════════════════════════════════════════════════════════════
NUISANCE_PARAMS = ['p', 's_mid', 's_hi']

def class_group(nuisance):
    if nuisance == 'p':     return [('p', 'hi'), ('p', 'mid')]
    if nuisance == 's_mid': return [('s', 'mid')]
    if nuisance == 's_hi':  return [('s', 'hi')]
    raise ValueError(nuisance)

def shift_class_dict(N_class_dict, nuisance, direction):
    sigma_table = SIGMA_REL_UP if direction == 'up' else SIGMA_REL_DOWN
    sign = +1 if direction == 'up' else -1
    return {
        cls: (n * (1.0 + sign * sigma_table[cls]) if cls in class_group(nuisance) else float(n))
        for cls, n in N_class_dict.items()
    }

# Baseline (fixed) totals -- computed once, held constant under any shift.
N_selected_FIXED = N_sig_TOTAL_reco + N_bkg_TOTAL_reco

def formula_v2(sig, bkg):
    """
    sig, bkg: possibly-shifted MATCHED class-count dicts.
    Unmatched counts are added back in unshifted (no sigma information).
    """
    N_bkg_shifted = sum(bkg.values()) + N_bkg_unmatched
    N_sig_shifted = sum(sig.values()) + N_sig_unmatched
    R_eff = N_sig_shifted / N_total_truth_FV
    return (N_selected_FIXED - N_bkg_shifted) / R_eff


def compute_xs_correlated_systematic(N_class_signal, N_class_bkg, formula_fn,
                                       verbose=True):
    central = formula_fn(N_class_signal, N_class_bkg)
    frac_up, frac_down = {}, {}
    for nuisance in NUISANCE_PARAMS:
        sig_up   = shift_class_dict(N_class_signal, nuisance, 'up')
        bkg_up   = shift_class_dict(N_class_bkg,    nuisance, 'up')
        sig_down = shift_class_dict(N_class_signal, nuisance, 'down')
        bkg_down = shift_class_dict(N_class_bkg,    nuisance, 'down')

        val_up   = formula_fn(sig_up,   bkg_up)
        val_down = formula_fn(sig_down, bkg_down)

        frac_up[nuisance]   = (val_up   - central) / central
        frac_down[nuisance] = (val_down - central) / central
        if verbose:
            print(f"  {nuisance:<6s}: {100*frac_up[nuisance]:+.2f}% / "
                  f"{100*frac_down[nuisance]:+.2f}%")

    total_up   = np.sqrt(sum(v**2 for v in frac_up.values()))
    total_down = np.sqrt(sum(v**2 for v in frac_down.values()))
    return central, total_up, total_down, {'up': frac_up, 'down': frac_down}


print("\n" + "=" * 70)
print("  CORRELATED SIGNAL + BACKGROUND SYSTEMATIC (reco-level, consistent)")
print("=" * 70)
central, total_up, total_down, per_nuisance = compute_xs_correlated_systematic(
    N_class_signal, N_class_bkg, formula_v2
)
print(f"\n  Central value = {central:.1f}  "
      f"(sanity check: should equal N_total_truth_FV = {N_total_truth_FV})")
print(f"  Total systematic = +{100*total_up:.2f}% / -{100*total_down:.2f}%")
print(f"  N_bkg / N_signal ratio (reco-level) = "
      f"{N_bkg_TOTAL_reco / N_sig_TOTAL_reco:.2f}")


# ═══════════════════════════════════════════════════════════════════════════
#  CELL 7 -- Signal-only alternative (no background term), for comparison /
#            as the fallback result while the background piece above is
#            still being validated
# ═══════════════════════════════════════════════════════════════════════════
delta_up_sq   = sum((N_class_signal[c] * SIGMA_REL_UP[c])**2   for c in CLASS_LABELS)
delta_down_sq = sum((N_class_signal[c] * SIGMA_REL_DOWN[c])**2 for c in CLASS_LABELS)
frac_up_sigonly   = np.sqrt(delta_up_sq)   / N_sig_TOTAL_reco
frac_down_sigonly = np.sqrt(delta_down_sq) / N_sig_TOTAL_reco

print("\n" + "=" * 70)
print("  SIGNAL-ONLY SYSTEMATIC (reco-level N', no background term)")
print("=" * 70)
print(f"  N' = {N_sig_TOTAL_reco}")
print(f"  Signal-only systematic: +{100*frac_up_sigonly:.2f}% / "
      f"-{100*frac_down_sigonly:.2f}%")

N_total (truth FV NCQE)              = 19439
N' (reco-selected FV signal)         = 897
N_bkg (reco-selected OOFV background) = 758

  CATEGORY BREAKDOWN (cat1 = at risk; cat2+cat3+cat4 = immune)
              cat1    cat2    cat3    cat4    dilution
  signal       670      16       7     204        2.6%
  bkg          566      28      17     147        5.9%

  CLASSIFIED COUNTS (cat1 / at-risk population only)
  Class         N (signal)     N (bkg)
  p-shell hi           544         483
  p-shell mid            0           0
  s-shell hi            56          38
  s-shell mid           70          45
  immune               227         192

  NOTE: immune events (cat2+cat3+cat4) are held FIXED -- no branching-ratio sigma applied -- since their selection does not depend on the primary de-excitation gamma. Only cat1 counts above move under the nuisance shifts below.

  CORRELATED SIGNAL + BACKGROUND SYSTEMATIC (reco-level, consistent)
  p     : -6.05% / +20.88%
  s_mid : -1.69% / +1.39%

In [ ]:
# Take the Correlated Signal + Background Systematic as our primary gamma-ray systematic
# (Total systematic = +6.35% / -20.95%)